In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

🚀 🔹 CELL 2 — LOAD DATA (PaySim)

In [4]:
df = pd.read_csv("PS_20174392719_1491204439457_log.csv")

# Select only relevant columns
df = df[['step','type','amount','oldbalanceOrg','newbalanceOrig',
         'oldbalanceDest','newbalanceDest','isFraud']]

print(df.head())
print(df['isFraud'].value_counts())

   step      type    amount  oldbalanceOrg  newbalanceOrig  oldbalanceDest  \
0     1   PAYMENT   9839.64       170136.0       160296.36             0.0   
1     1   PAYMENT   1864.28        21249.0        19384.72             0.0   
2     1  TRANSFER    181.00          181.0            0.00             0.0   
3     1  CASH_OUT    181.00          181.0            0.00         21182.0   
4     1   PAYMENT  11668.14        41554.0        29885.86             0.0   

   newbalanceDest  isFraud  
0             0.0        0  
1             0.0        0  
2             0.0        1  
3             0.0        1  
4             0.0        0  
isFraud
0    6354407
1       8213
Name: count, dtype: int64


🚀 🔹 CELL 3 — PREPROCESSING

In [5]:
# Encode transaction type
le = LabelEncoder()
df['type'] = le.fit_transform(df['type'])

# Features & Target
X = df.drop('isFraud', axis=1)
y = df['isFraud']

🚀 🔹 CELL 4 — TRAIN TEST SPLIT

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

🚀 🔹 CELL 5 — MODEL PERFORMANCE COMPARISON (ONE CELL)

In [7]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "XGBoost": XGBClassifier(eval_metric='logloss'),
    "LightGBM": LGBMClassifier()
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]
    
    results.append({
        "Model": name,
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results)
print("\nMODEL PERFORMANCE COMPARISON:\n")
print(results_df.sort_values(by="F1 Score", ascending=False))

[LightGBM] [Info] Number of positive: 6570, number of negative: 5083526
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.146072 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1535
[LightGBM] [Info] Number of data points in the train set: 5090096, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001291 -> initscore=-6.651247
[LightGBM] [Info] Start training from score -6.651247

MODEL PERFORMANCE COMPARISON:

                 Model  Precision    Recall  F1 Score   ROC-AUC
1        Random Forest   0.977980  0.783932  0.870270  0.995890
2              XGBoost   0.884159  0.771150  0.823797  0.961686
0  Logistic Regression   0.901048  0.471089  0.618705  0.968442
3             LightGBM   0.238128  0.415094  0.302640  0.615061


🚀 🔹 CELL 6 — TRAIN BEST MODEL (XGBoost)

In [ ]:
best_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    eval_metric='logloss'
)

best_model.fit(X_train, y_train)

🚀 🔹 CELL 7 — FINAL EVALUATION

In [ ]:
from sklearn.metrics import classification_report

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

🚀 🔥 CELL 8 — REAL-TIME PREDICTION FUNCTION (UPDATED)

In [ ]:
def predict_realtime(step, type_, amount,
                     oldbalanceOrg, newbalanceOrig,
                     oldbalanceDest, newbalanceDest):
    
    # Validate type input
    if type_ not in le.classes_:
        print(f"❌ Invalid type. Allowed values: {list(le.classes_)}")
        return
    
    type_encoded = le.transform([type_])[0]
    
    data = pd.DataFrame([[
        step,
        type_encoded,
        amount,
        oldbalanceOrg,
        newbalanceOrig,
        oldbalanceDest,
        newbalanceDest
    ]], columns=X.columns)
    
    pred = best_model.predict(data)[0]
    prob = best_model.predict_proba(data)[0][1]
    
    if pred == 1:
        print(f"⚠️ FRAUD DETECTED (Confidence: {prob:.4f})")
    else:
        print(f"✅ NORMAL TRANSACTION (Confidence: {1 - prob:.4f})")

🚀 🔹 CELL 9 — TEST REAL-TIME INPUT

In [ ]:
predict_realtime(
    step=1,
    type_='TRANSFER',   # Must be one of dataset values
    amount=50000,
    oldbalanceOrg=100000,
    newbalanceOrig=50000,
    oldbalanceDest=0,
    newbalanceDest=50000
)

🎯 INPUT FORMAT (FOR YOUR REPORT)
✔ Model Inputs:
Feature	Description

1.step	          = Time step (hour)
2.type	          = Transaction type
3.amount	      = Transaction amount
4.oldbalanceOrg	  = Sender balance before
5.newbalanceOrig  = Sender balance after
6.oldbalanceDest  = Receiver balance before
7.newbalanceDest  = Receiver balance after

In [ ]:
! pip install catboost shap

In [ ]:
from catboost import CatBoostClassifier
import shap

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "XGBoost": XGBClassifier(eval_metric='logloss'),
    "LightGBM": LGBMClassifier(),
    "CatBoost": CatBoostClassifier(verbose=0)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]
    
    results.append({
        "Model": name,
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results)
print("\nMODEL PERFORMANCE COMPARISON:\n")
print(results_df.sort_values(by="F1 Score", ascending=False))

In [ ]:
best_model = CatBoostClassifier(
    iterations=200,
    depth=6,
    learning_rate=0.1,
    verbose=0
)

best_model.fit(X_train, y_train)

In [ ]:
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

In [ ]:
shap.summary_plot(shap_values, X_test)

In [ ]:
# Select one sample
sample = X_test.iloc[0]

shap.force_plot(
    explainer.expected_value,
    shap_values[0],
    sample
)

In [ ]:
def predict_with_explanation(step, type_, amount,
                            oldbalanceOrg, newbalanceOrig,
                            oldbalanceDest, newbalanceDest):
    
    if type_ not in le.classes_:
        print(f"❌ Invalid type. Allowed: {list(le.classes_)}")
        return
    
    type_encoded = le.transform([type_])[0]
    
    data = pd.DataFrame([[
        step,
        type_encoded,
        amount,
        oldbalanceOrg,
        newbalanceOrig,
        oldbalanceDest,
        newbalanceDest
    ]], columns=X.columns)
    
    pred = best_model.predict(data)[0]
    prob = best_model.predict_proba(data)[0][1]
    
    # SHAP explanation
    shap_val = explainer.shap_values(data)
    
    print("\n🔍 Prediction Result:")
    if pred == 1:
        print(f"⚠️ FRAUD DETECTED (Confidence: {prob:.4f})")
    else:
        print(f"✅ NORMAL TRANSACTION (Confidence: {1-prob:.4f})")
    
    print("\n📊 Feature Contribution (XAI):")
    for feature, value in zip(X.columns, shap_val[0]):
        print(f"{feature}: {value:.4f}")

In [ ]:
predict_with_explanation(
    step=1,
    type_='TRANSFER',
    amount=50000,
    oldbalanceOrg=100000,
    newbalanceOrig=50000,
    oldbalanceDest=0,
    newbalanceDest=50000
)